# Suite 01 — S+ / S- / O+ / Other across all NWBs

Reproducible classification suite:
1. Freeze shuffle-controlled config (seed, FDR, effect floors).
2. Discover every `*.nwb` under the data root.
3. Classify **NWB-by-NWB**.
4. Write per-session tables and **append/replace** into the grand unit table.
5. Summarize prevalences (overall, by session, by area).

**Definitions** (`jnwb.unit_classification`):
- **S+ / S-**: present-stimulus slots across 12 GLO conditions vs fx baseline; one-sided shuffle + session BH-FDR.
- **O+**: omission slots vs local baseline **and** matched control **and** mean(d1–d4) on the same trial (anti-fatigue); all shuffle+FDR at q&lt;0.01; priority over S+/S-.
- **Other**: not significant under the above.

**Outputs**:
- `outputs/classification/grand_unit_table_shuffle_sso.csv`
- `outputs/classification/grand_unit_table_shuffle_sso.meta.json`
- `outputs/classification/per_session/{stem}_shuffle_class.csv`
- `outputs/classification/classification_config_freeze.json`

## 0. Setup

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from jnwb.unit_classification import (
    ClassificationConfig,
    append_session_to_grand_table,
    classify_all_nwbs,
    classify_nwb_file,
    config_to_dict,
    discover_nwb_paths,
    prevalence_summary,
)

pd.set_option("display.max_rows", 40)
pd.set_option("display.width", 120)
print("ROOT =", ROOT)

ROOT = D:\workspace\omission


## 1. Frozen config (reproducibility contract)

Edit only deliberately — changing these changes labels.

In [2]:
NWB_ROOT = Path(r"D:/analysis/nwb")
OUT_DIR = ROOT / "outputs" / "classification"
GRAND_PATH = OUT_DIR / "grand_unit_table_shuffle_sso.csv"
PER_SESSION_DIR = OUT_DIR / "per_session"

CFG = ClassificationConfig(
    n_shuffles=1000,
    alpha=0.05,
    alpha_omission=0.01,
    seed=42,
    apply_fdr=True,
    min_abs_stim_effect_hz=0.5,
    min_omission_effect_hz=2.0,
    min_baseline_for_s_minus_hz=3.5,
    min_stim_rate_for_s_plus_hz=0.5,
)

# Smoke: set to 1 to process only the first NWB. None = full catalog.
MAX_FILES = None

OUT_DIR.mkdir(parents=True, exist_ok=True)
PER_SESSION_DIR.mkdir(parents=True, exist_ok=True)

print("NWB_ROOT:", NWB_ROOT)
print("GRAND_PATH:", GRAND_PATH)
print("MAX_FILES:", MAX_FILES)
print(json.dumps(config_to_dict(CFG), indent=2))

NWB_ROOT: D:\analysis\nwb
GRAND_PATH: D:\workspace\omission\outputs\classification\grand_unit_table_shuffle_sso.csv
MAX_FILES: None
{
  "n_shuffles": 1000,
  "alpha": 0.05,
  "alpha_omission": 0.01,
  "min_trials": 8,
  "min_stim_events": 20,
  "min_omission_events": 8,
  "seed": 42,
  "apply_fdr": true,
  "min_abs_stim_effect_hz": 0.5,
  "min_omission_effect_hz": 2.0,
  "min_baseline_for_s_minus_hz": 3.5,
  "min_stim_rate_for_s_plus_hz": 0.5,
  "method": "shuffle_controlled_sso_v2_omit_vs_delay",
  "glo_conditions": [
    "AAAB",
    "AXAB",
    "AAXB",
    "AAAX",
    "BBBA",
    "BXBA",
    "BBXA",
    "BBBX",
    "RRRR",
    "RXRR",
    "RRXR",
    "RRRX"
  ]
}


## 2. Discover NWB files

In [3]:
nwb_paths = discover_nwb_paths(NWB_ROOT)
print(f"Found {len(nwb_paths)} NWB files")
for p in nwb_paths:
    print(" ", p.name)

Found 15 NWB files
  sub-C31o_ses-230816_rec.nwb
  sub-C31o_ses-230818_rec.nwb
  sub-C31o_ses-230823_rec.nwb
  sub-C31o_ses-230825_rec.nwb
  sub-C31o_ses-230830_rec.nwb
  sub-C31o_ses-230831_rec.nwb
  sub-C31o_ses-230901_rec.nwb
  sub-V182o_ses-260629.nwb
  sub-V182o_ses-260702.nwb
  sub-V182o_ses-260706.nwb
  sub-V182o_ses-260708.nwb
  sub-V198o_ses-230714_rec.nwb
  sub-V198o_ses-230719_rec.nwb
  sub-V198o_ses-230720_rec.nwb
  sub-V198o_ses-230721_rec.nwb


## 3. Classify all sessions → grand table

Idempotent: re-running **replaces** rows for the same `nwb_stem`.
Each session also writes `per_session/{stem}_shuffle_class.csv`.

In [4]:
grand = classify_all_nwbs(
    nwb_root=NWB_ROOT,
    grand_path=GRAND_PATH,
    cfg=CFG,
    per_session_dir=PER_SESSION_DIR,
    max_files=MAX_FILES,
)
print(f"Grand table rows: {len(grand)}")
print(f"Sessions: {grand['nwb_stem'].nunique() if len(grand) else 0}")
print("Wrote:", GRAND_PATH)
print("Meta:", GRAND_PATH.with_suffix(".meta.json"))

2026-07-11 14:00:10,553 - INFO - [1/15] classifying sub-C31o_ses-230816_rec.nwb


2026-07-11 14:00:10,826 - INFO - ✓ Loaded cached NWB tables for sub-C31o_ses-230816_rec from disk cache


2026-07-11 14:00:10,826 - INFO - ✓ Loaded sub-C31o_ses-230816_rec.nwb


2026-07-11 14:00:11,208 - INFO - onsets AAAB: n=246


2026-07-11 14:00:11,209 - INFO - onsets AXAB: n=34


...

2026-07-11 14:12:20,742 - INFO - classified 150 / 277 units


2026-07-11 14:12:26,130 - INFO - classified 200 / 277 units


2026-07-11 14:12:31,512 - INFO - classified 250 / 277 units


2026-07-11 14:12:34,478 - INFO -   sub-V198o_ses-230721_rec prevalence {'S+': 0.08303249097472924, 'S-': 0.07581227436823104, 'O+': 0.0, 'Other': 0.8411552346570397, 'n_units': 277.0, 'n_S+': 23.0, 'n_S-': 21.0, 'n_O+': 0.0, 'n_Other': 233.0}


Grand table rows: 6655
Sessions: 15
Wrote: D:\workspace\omission\outputs\classification\grand_unit_table_shuffle_sso.csv
Meta: D:\workspace\omission\outputs\classification\grand_unit_table_shuffle_sso.meta.json


## 4. Prevalence summary (overall)

In [5]:
if len(grand):
    prev = prevalence_summary(grand)
    print(json.dumps(prev, indent=2))
    print("\nCounts:")
    print(grand["display_class"].value_counts().to_string())
else:
    print("Grand table empty — check NWB_ROOT / errors above.")

{
  "S+": 0.21517655897821186,
  "S-": 0.1138993238166792,
  "O+": 0.0010518407212622089,
  "Other": 0.6698722764838467,
  "n_units": 6655.0,
  "n_S+": 1432.0,
  "n_S-": 758.0,
  "n_O+": 7.0,
  "n_Other": 4458.0
}

Counts:
display_class
Other    4458
S+       1432
S-        758
O+          7


## 5. Prevalence by session

In [6]:
if len(grand):
    rows = []
    for stem, g in grand.groupby("nwb_stem"):
        p = prevalence_summary(g)
        p["nwb_stem"] = stem
        rows.append(p)
    by_ses = pd.DataFrame(rows).set_index("nwb_stem")
    display_cols = ["n_units", "S+", "S-", "O+", "Other", "n_S+", "n_S-", "n_O+"]
    print(by_ses[display_cols].sort_index().to_string())
    by_ses.to_csv(OUT_DIR / "prevalence_by_session.csv")
    print("Wrote", OUT_DIR / "prevalence_by_session.csv")

                          n_units        S+        S-        O+     Other   n_S+   n_S-  n_O+
nwb_stem                                                                                     
sub-C31o_ses-230816_rec     357.0  0.243697  0.170868  0.000000  0.585434   87.0   61.0   0.0
sub-C31o_ses-230818_rec     541.0  0.304991  0.147874  0.000000  0.547135  165.0   80.0   0.0
sub-C31o_ses-230823_rec     368.0  0.228261  0.320652  0.019022  0.432065   84.0  118.0   7.0
sub-C31o_ses-230825_rec     491.0  0.134420  0.199593  0.000000  0.665988   66.0   98.0   0.0
sub-C31o_ses-230830_rec     774.0  0.091731  0.038760  0.000000  0.869509   71.0   30.0   0.0
sub-C31o_ses-230831_rec     584.0  0.166096  0.147260  0.000000  0.686644   97.0   86.0   0.0
sub-C31o_ses-230901_rec     696.0  0.100575  0.080460  0.000000  0.818966   70.0   56.0   0.0
sub-V182o_ses-260629        293.0  0.215017  0.174061  0.000000  0.610922   63.0   51.0   0.0
sub-V182o_ses-260702        409.0  0.220049  0.156479  0.000

## 6. Class × area (O+ enrichment sanity check)

In [7]:
if len(grand):
    ct = pd.crosstab(grand["area"].astype(str), grand["display_class"])
    for col in ("S+", "S-", "O+", "Other"):
        if col not in ct.columns:
            ct[col] = 0
    ct["n"] = ct.sum(axis=1)
    ct["O+_frac"] = ct["O+"] / ct["n"].clip(lower=1)
    print(ct[["n", "S+", "S-", "O+", "Other", "O+_frac"]].sort_values("O+_frac", ascending=False).to_string())
    ct.to_csv(OUT_DIR / "class_by_area.csv")

display_class     n   S+   S-  O+  Other   O+_frac
area                                              
FEF             771  195  118   6    452  0.007782
V1             1267  513  127   1    626  0.000789
MST/FST          36    9    0   0     27  0.000000
PFC            1342  164   84   0   1094  0.000000
MT              672  170   41   0    461  0.000000
TEO             459   61   98   0    300  0.000000
V3              329   95   81   0    153  0.000000
V3d             649   60   40   0    549  0.000000
V4             1130  165  169   0    796  0.000000


## 7. Trace one unit from the grand table

1. Note `nwb_stem` + `unit_id`.
2. Open `per_session/{stem}_shuffle_class.csv` for full p/q/effect columns.
3. Confirm `cfg_*`, `git_sha`, `method` match `classification_config_freeze.json`.
4. Raster: `jnwb.viz.raster_suite_omission(session, unit_id=...)`.

In [8]:
if len(grand):
    example = (
        grand.sort_values(["display_class", "nwb_stem", "unit_id"])
        .groupby("display_class", sort=False)
        .head(2)
    )
    cols = [
        c
        for c in [
            "nwb_stem",
            "unit_id",
            "area",
            "display_class",
            "stim_effect_hz",
            "q_s_plus_shuffle",
            "q_s_minus_shuffle",
            "om_vs_base_effect_hz",
            "q_om_vs_base_shuffle",
            "q_om_vs_ctrl_shuffle",
            "git_sha",
            "method",
        ]
        if c in grand.columns
    ]
    print(example[cols].to_string(index=False))

               nwb_stem  unit_id area display_class  stim_effect_hz  q_s_plus_shuffle  q_s_minus_shuffle  om_vs_base_effect_hz  q_om_vs_base_shuffle  q_om_vs_ctrl_shuffle git_sha                                  method
sub-C31o_ses-230823_rec       41  FEF            O+        6.106290          0.003404           1.000000              4.743052              0.007208              0.004275 c8efc80 shuffle_controlled_sso_v2_omit_vs_delay
sub-C31o_ses-230823_rec       49  FEF            O+        3.036888          0.003404           1.000000              6.588089              0.007208              0.004275 c8efc80 shuffle_controlled_sso_v2_omit_vs_delay
sub-C31o_ses-230816_rec        2  PFC         Other        0.135220          0.003213           1.000000              0.077933              0.273174              1.000000 c8efc80 shuffle_controlled_sso_v2_omit_vs_delay
sub-C31o_ses-230816_rec        3  PFC         Other        0.038245          0.070739           1.000000              0.0042

## 8. Optional: reclassify one file only

In [9]:
# Uncomment to refresh a single session in the grand table:
# one = nwb_paths[0]
# sdf = classify_nwb_file(one, cfg=CFG)
# append_session_to_grand_table(sdf, GRAND_PATH, CFG)
# print(prevalence_summary(sdf))